<a href="https://colab.research.google.com/github/GelsonRibeiroJr/alura-agent-rag/blob/main/1%C2%BA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Importando Bibliotecas*

In [ ]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph

Chave de API - GROQ & NASA



In [ ]:
from google.colab import userdata
import os

def carregar_chave(nome_secret):
    valor = userdata.get(nome_secret)
    if not valor:
        raise ValueError(f"❌ Secret '{nome_secret}' não encontrada ou vazia. Configure em Colab > Secrets.")
    os.environ[nome_secret] = valor
    return valor

# Carrega a chave da Groq
groq_api_key = carregar_chave('GROQ_API_KEY')

# Carrega a chave da NASA
nasa_api_key = carregar_chave('NASA_API_KEY')

# Carrega o token do Hugging Face
hf_token = carregar_chave('HF_TOKEN')

print("✅ Chaves de API carregadas com sucesso!")

✅ Chaves de API carregadas com sucesso!


Cloando o repositorio para o ambiente Colab

In [ ]:
!git clone https://github.com/GelsonRibeiroJr/alura-agent-rag.git

fatal: destination path 'alura-agent-rag' already exists and is not an empty directory.


In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Caminho apontando para a pasta baixada do GitHub
path_data = './alura-agent-rag/data'

# Configura o leitor para processar todos os PDFs
loader = DirectoryLoader(
    path_data,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

# Carrega todos os documentos
documents = loader.load()

print(f"Sucesso! Total de páginas processadas: {len(documents)}")

100%|██████████| 29/29 [00:41<00:00,  1.42s/it]

Sucesso! Total de páginas processadas: 400


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configura o fatiador de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Tamanho aproximado de cada pedaço (caracteres)
    chunk_overlap=150  # Quantidade de caracteres sobrepostos
)

# Aplica a divisão em todas as 400 páginas
chunks = text_splitter.split_documents(documents)

print(f"Base fatiada com sucesso! Total de chunks gerados: {len(chunks)}")
print("\n--- Exemplo do primeiro Chunk gerado ---")
print(chunks[0].page_content[:300]) # Mostra os primeiros 300 caracteres do 1º pedaço

Base fatiada com sucesso! Total de chunks gerados: 890

--- Exemplo do primeiro Chunk gerado ---
National Aeronautics and Space Administration
Geology Training for Artemis Missions
National Academies Panel on Lunar and 
Planetary Sciences for Key Non-Polar 
Destinations Across the Moon to Address 
Decadal-level Science Objectives with 
Human Explorers 
Cynthia Evans, Ph.D. 
Artemis Geology Trai


In [ ]:

# --- CRIAÇÃO DA MEMÓRIA PERSISTENTE com SQLite
%pip install -qU langgraph-checkpoint-sqlite

# Instala a biblioteca de embeddings da HuggingFace e o banco vetorial FAISS
%pip install -q sentence-transformers faiss-cpu

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Carregando o modelo de Embeddings...")
# Usamos um modelo multilíngue super eficiente e leve
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print(f"Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

# Salva o índice localmente para não precisar reprocessar tudo se reiniciar o ambiente
vectorstore.save_local("faiss_index")

print(f"✅ Banco Vetorial criado com sucesso! Todos os {len(chunks)} chunks foram indexados.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 8.9 MB/s eta 0:00:00
Carregando o modelo de Embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.
✅ Banco Vetorial criado com sucesso! Todos os 890 chunks foram indexados.


In [ ]:
# Teste de busca por similaridade semântica
pergunta = "Qual é o objetivo do programa Human Landing System no projeto Artemis?"

# Busca os 3 pedaços de texto mais parecidos no banco
resultados = vectorstore.similarity_search(pergunta, k=3)

print("--- Trechos mais relevantes encontrados pelo Banco Vetorial ---")
for i, doc in enumerate(resultados):
    print(f"\nResultado {i+1}:")
    print(doc.page_content[:400]) # Exibe os 400 primeiros caracteres do resultado

--- Trechos mais relevantes encontrados pelo Banco Vetorial ---

Resultado 1:
AAS 23-057
AN OVERVIEW OF THE ARTEMIS I NAVIGATION PERFORMANCE
Greg Holt*, Chris D’Souza †, and Michael Wasinger ‡
The goal of NASA’s Artemis Program is to explore the Moon and beyond. The
Artemis I Mission which flew in late 2022 was the uncrewed test flight whose goal
was to exercise the entire navigation system in an extended duration flight and
evaluate its performance over the entire mission,

Resultado 2:
Artemis program will land the first woman and next man on the surface of the Moon and 
establish, together with international and commercial partners, the sustainable human exploration 
of the solar system; 
 
CONSIDERING the necessity of greater coordination and cooperation between and among 
established and emerging actors in space; 
 
RECOGNIZING the global benefits of space exploration and com

Resultado 3:
11
Chapter 1: Setting Humanity on a Sustainable Course  
to the Moon
The Artemis program bui

Importando **ChatGroq**

In [ ]:
from langchain_groq import ChatGroq

# Inicializa o modelo LLM da Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2, # Baixa temperatura para manter fidelidade aos documentos
    api_key=os.environ.get("GROQ_API_KEY")
)

# Teste rápido direto do modelo (sem RAG)
resposta_teste = llm.invoke("Diga 'Modelo Groq conectado com sucesso!' em português.")
print(resposta_teste.content)

"Modelo Groq conectado com sucesso!"


Configurando o Agente ReAct (Tools + Guardrail de Escopo)

In [ ]:
import sqlite3
import requests
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver

# --- 1. DEFINIÇÃO DAS FERRAMENTAS (@tool) ---

@tool
def pega_contexto_artemis_lunar(query: str) -> str:
    """Busca contexto técnico sobre o Programa Artemis, naves, rotas e missões na Lua nos PDFs oficiais da NASA."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)

    texto_formatado = []
    for doc in resultado:
        fonte = doc.metadata.get('source', 'NASA').split('/')[-1]
        pagina = doc.metadata.get('page', 0) + 1
        texto_formatado.append(f"[Documento: '{fonte}', Página {pagina}]\n{doc.page_content}")

    return "\n\n---\n\n".join(texto_formatado)

@tool
def consulta_api_nasa(query: str) -> str:
    """Consulta a API da NASA para dúvidas sobre projetos gerais, telescópios ou espaço que não estão nos PDFs."""
    try:
        url = f"https://images-api.nasa.gov/search?q={query}&media_type=image"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            items = response.json().get("collection", {}).get("items", [])
            if items:
                desc = items[0]["data"][0].get("description", "")
                return f"[Fonte: API da NASA]\n{desc[:500]}"
    except Exception:
        pass
    return "Nenhum dado retornado da API da NASA."

tools = [pega_contexto_artemis_lunar, consulta_api_nasa]


# --- 2. PROMPT DO AGENTE  ---
system_prompt = """Você é um assistente técnico especialista da NASA.
Responda à pergunta do usuário de forma DIRETA, OBJETIVA e em Português do Brasil.
Não faça justificativas, introduções ou explicações sobre a busca.

REGRA OBRIGATÓRIA DE CITAÇÃO DE FONTE (NÃO IGNORE):
Toda resposta baseada em ferramentas DEVE seguir o formato de saída abaixo.
- A ferramenta 'pega_contexto_artemis_lunar' retorna o texto com a tag [Documento: 'NOME', Página X]. Você DEVE copiar essa informação exata para a última linha.
- A ferramenta 'consulta_api_nasa' retorna a tag [Fonte: API da NASA].

FORMATO DE SAÍDA OBRIGATÓRIO:
[Sua resposta direta e objetiva aqui]

Fonte: [Insira a citação exata aqui]

REGRA DE FORA DE ESCOPO (MUITO IMPORTANTE):
Se o usuário fizer perguntas que NÃO tenham relação com espaço, astronomia ou NASA (exemplo: esportes, culinária, política, etc.), NÃO USE NENHUMA FERRAMENTA. Responda APENAS:
"Sou um agente especialista em responder perguntas sobre a exploração lunar e sobre a NASA. Sua pergunta está fora do meu escopo de conhecimento."
"""

# --- 3. CRIAÇÃO DO AGENTE COM MEMÓRIA PERSISTENTE (SQLite) ---

conn = sqlite3.connect("memoria_agente.db", check_same_thread=False)
memoria = SqliteSaver(conn)
memoria.setup()
# Criação do agente
agente_nasa = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memoria
)

# --- 4. FUNÇÃO DE CHAT COM CONTEXTO ---
def chat_com_memoria(mensagem_usuario: str, thread_id="1"):
    """Função para enviar mensagens ao agente mantendo o histórico da conversa."""
    config = {"configurable": {"thread_id": thread_id}}
    resposta = agente_nasa.invoke({"messages": [("user", mensagem_usuario)]}, config)
    return resposta["messages"][-1].content

print("✅ Agente da NASA configurado com sucesso!")

✅ Agente da NASA configurado com sucesso!


Testes do Funcionamento

In [ ]:
print("=== TESTE DE MEMÓRIA DO AGENTE NASA ===\n")

# Pergunta 1: O agente busca nos PDFs e responde
print(" Usuário: Qual foi o objetivo da missão Artemis I?")
resp1 = chat_com_memoria("Qual foi o objetivo da missão Artemis I?", thread_id="teste_memoria")
print(f" Agente:\n{resp1}\n")
print("-" * 50 + "\n")


=== TESTE DE MEMÓRIA DO AGENTE NASA ===

 Usuário: Qual foi o objetivo da missão Artemis I?
 Agente:
O objetivo da missão Artemis I foi testar o sistema de escape do veículo espacial Orion e verificar sua capacidade de sobreviver à reentrada na atmosfera da Terra após uma viagem ao espaço, além de exercitar o sistema de navegação do veículo espacial em uma viagem de longa duração.

Fonte: [Documento: 'AAS-23-057_Artemis_1_Navigation_HoltDsouzaWasinger_FINALfinal.pdf', Página 1]

--------------------------------------------------



In [ ]:
# Pergunta 2: O agente precisa lembrar do assunto (Artemis I) sem que o nome seja citado
print(" Usuário: E quando ela foi lançada?")
resp2 = chat_com_memoria("E quando ela foi lançada?", thread_id="teste_memoria")
print(f"Agente:\n{resp2}\n")

 Usuário: E quando ela foi lançada?
Agente:
A missão Artemis I foi lançada em 16 de novembro de 2022.

Fonte: [Documento: 'AAS-23-057_Artemis_1_Navigation_HoltDsouzaWasinger_FINALfinal.pdf', Página 1]



In [ ]:
# --- TESTE 2: Pergunta Geral sobre Espaço (API) ---
print("=== TESTE 2: BUSCA FORA DOS DOCS (API DA NASA) ===")
resposta_api = chat_com_memoria("O que é o telescópio espacial Hubble?", thread_id="api_test_thread")
print(resposta_api)
print("\n" + "-"*50 + "\n")

=== TESTE 2: BUSCA FORA DOS DOCS (API DA NASA) ===
O telescópio espacial Hubble é um observatório espacial que foi lançado em 1990 e é operado pela NASA. Ele é projetado para observar o universo em diferentes comprimentos de onda, desde raios gama até ondas de rádio, e tem sido fundamental para a compreensão da formação e evolução do universo.

O Hubble é capaz de capturar imagens de alta resolução de objetos distantes, como galáxias, estrelas e planetas, e tem feito contribuições importantes para a astronomia, incluindo a medição da expansão do universo e a descoberta de exoplanetas.

O Hubble é um dos observatórios espaciais mais bem-sucedidos da história e tem sido objeto de estudo e admiração por cientistas e entusiastas de todo o mundo.

Fonte: [Fonte: API da NASA]

--------------------------------------------------



In [ ]:
# --- TESTE 3: Pergunta Fora de Escopo (Guardrail) ---
print("=== TESTE 3: FORA DE ESCOPO (GUARDRAIL) ===")
# Testando com um tema aleatório para ver se a trava funciona para qualquer assunto
pergunta_aleatoria = "Qual é a melhor forma de harmonizar um vinho tinto ou fazer um bolo de chocolate?"
resposta_fora_escopo = chat_com_memoria(pergunta_aleatoria, thread_id="guardrail_test_thread")
print(f"Pergunta do usuário: {pergunta_aleatoria}")
print(f"Resposta do Agente: {resposta_fora_escopo}")
print("\n" + "-"*50 + "\n")

=== TESTE 3: FORA DE ESCOPO (GUARDRAIL) ===
Pergunta do usuário: Qual é a melhor forma de harmonizar um vinho tinto ou fazer um bolo de chocolate?
Resposta do Agente: Sou um agente especialista em responder perguntas sobre a exploração lunar e sobre a NASA. Sua pergunta está fora do meu escopo de conhecimento.

--------------------------------------------------

